<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
!pip3 install beautifulsoup4
!pip3 install requests

In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
# use requests.get() method with the provided static_url
# assign the response to a object
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(url)

if response.status_code == 200:
    print("Successfully retrieved the page")
else:
    print("Failed to retrieve the page")

Successfully retrieved the page


In [8]:
import requests
from bs4 import BeautifulSoup

# Step 1: Fetch the HTML content
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(url)

# Step 2: Check the response status
if response.status_code == 200:
    print("Successfully retrieved the page")
    
    # Step 3: Create a BeautifulSoup object
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # You can now use 'soup' to navigate and search the HTML tree
   # print(soup.prettify())  # Optional: Print the formatted HTML
else:
    print("Failed to retrieve the page")

Successfully retrieved the page


Create a `BeautifulSoup` object from the HTML `response`


Print the page title to verify if the `BeautifulSoup` object was created properly 


In [9]:
# Use soup.title attribute
print(soup.title.string)

List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


Starting from the third table is our target table contains the actual launch records.


In [1]:
import requests
from bs4 import BeautifulSoup

# 指定维基百科页面的 URL
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

# 发送 GET 请求
response = requests.get(url)

# 检查请求是否成功
if response.status_code == 200:
    # 创建 BeautifulSoup 对象
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 找到所有的表格
    tables = soup.find_all('table', {'class': 'wikitable'})
    
    # 打印找到的表格数量
    print(f"找到的表格数量: {len(tables)}")
    
    # 选择第三个表格（索引为2）
    target_table = tables[2]  # 从第三个表格开始
    
    # 提取列名
    headers = [header.text.strip() for header in target_table.find_all('th')]
    
    # 打印列名
    print("列名:")
    for header in headers:
        print(header)
else:
    print(f"请求失败，状态码: {response.status_code}")

找到的表格数量: 13
列名:
Flight No.
Date andtime (UTC)
Version,Booster[b]
Launch site
Payload[c]
Payload mass
Orbit
Customer
Launchoutcome
Boosterlanding
14
15
16
17
18
19
20


In [19]:
# Let's print the third table and check its content
first_launch_table = tables[2]
print(first_launch_table)

<table class="wikitable plainrowheaders collapsible" style="width: 100%;">
<tbody><tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11"><span class="cite-bracket">[</span>b<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12"><span class="cite-bracket">[</span>c<span class="cite-bracket">]</span></a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


In [21]:
column_names = []

# 找到第一个表格
first_launch_table = tables[2] # 假设你想要第一个表格

# 使用 find_all() 函数找到所有 th 元素
headers = first_launch_table.find_all('th')

# 迭代每个 th 元素并提取列名
for header in headers:
    column_name = extract_column_from_header(header)
    if column_name:  # 仅添加非空列名
        column_names.append(column_name)

print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [22]:
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [23]:
launch_dict= dict.fromkeys(column_names)

# Remove an irrelvant column
del launch_dict['Date and time ( )']

# Let's initial the launch_dict with each value to be an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 辅助函数定义（示例）
def date_time(date_time_str):
    # 假设这个函数将日期和时间字符串分开
    return date_time_str.split()  # 示例：按空格分割

def booster_version(version_str):
    # 假设这个函数处理火箭版本
    return version_str.strip()  # 示例：去除空格

def get_mass(mass_str):
    # 假设这个函数从字符串中提取有效的质量
    return mass_str.strip()  # 示例：去除空格

def landing_status(status_str):
    # 假设这个函数处理着陆状态
    return status_str.strip()  # 示例：去除空格

# 指定维基百科页面的 URL
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

# 发送 GET 请求
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 创建初始字典
    launch_dict = {
        'Flight No.': [],
        'Date': [],
        'Time': [],
        'Version, Booster': [],
        'Launch site': [],
        'Payload': [],
        'Payload mass': [],
        'Orbit': [],
        'Customer': [],
        'Launch outcome': [],
        'Booster landing': []
    }
    
    extracted_row = 0
    
    # 提取每个表格
    for table in soup.find_all('table', "wikitable plainrowheaders collapsible"):
        # 获取表格行
        for rows in table.find_all("tr"):
            # 检查第一列是否为数字
            if rows.th:
                if rows.th.string:
                    flight_number = rows.th.string.strip()
                    flag = flight_number.isdigit()
                else:
                    flag = False
            else:
                flag = False
            
            # 获取表格元素
            row = rows.find_all('td')
            # 如果是数字，保存单元格到字典中
            if flag:
                extracted_row += 1
                
                # Flight Number value
                launch_dict['Flight No.'].append(flight_number)
                
                # Date and Time values
                datatimelist = date_time(row[0].text)
                
                # Date value
                date = datatimelist[0].strip(',')
                launch_dict['Date'].append(date)
                
                # Time value
                time = datatimelist[1]
                launch_dict['Time'].append(time)
                
                # Booster version
                bv = booster_version(row[1].text)
                if not bv:
                    bv = row[1].a.string if row[1].a else ''
                launch_dict['Version, Booster'].append(bv)
                
                # Launch Site
                launch_site = row[2].a.string if row[2].a else ''
                launch_dict['Launch site'].append(launch_site)
                
                # Payload
                payload = row[3].a.string if row[3].a else ''
                launch_dict['Payload'].append(payload)
                
                # Payload Mass
                payload_mass = get_mass(row[4].text)
                launch_dict['Payload mass'].append(payload_mass)
                
                # Orbit
                orbit = row[5].a.string if row[5].a else ''
                launch_dict['Orbit'].append(orbit)
                
                # Customer
                customer = row[6].a.string if row[6].a else ''
                launch_dict['Customer'].append(customer)
                
                # Launch outcome
                launch_outcome = list(row[7].strings)[0] if row[7].strings else ''
                launch_dict['Launch outcome'].append(launch_outcome)
                
                # Booster landing
                booster_landing = landing_status(row[8].text)
                launch_dict['Booster landing'].append(booster_landing)

    # 打印结果
    print("填充后的字典:")
    print(launch_dict)

else:
    print(f"请求失败，状态码: {response.status_code}")

填充后的字典:
{'Flight No.': ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121'], 'Date': ['4', '8', '22', '8', '1', '29', '3', '6', '18', '14', '5', '7', '21', '10', '11', '2', '14', '27', '28', '22', '17', '4', '8', '6', '27', '15', '18', '14', '14', '19', '16', '30', '1', '15', '3', '23', '25', '5', '14', '24', '7

To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


## Authors


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 辅助函数定义（示例）
def date_time(date_time_str):
    return date_time_str.split()  # 示例：按空格分割

def booster_version(version_str):
    return version_str.strip()  # 示例：去除空格

def get_mass(mass_str):
    return mass_str.strip()  # 示例：去除空格

def landing_status(status_str):
    return status_str.strip()  # 示例：去除空格

# 指定维基百科页面的 URL
url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

# 发送 GET 请求
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 创建初始字典
    launch_dict = {
        'Flight No.': [],
        'Date': [],
        'Time': [],
        'Version, Booster': [],
        'Launch site': [],
        'Payload': [],
        'Payload mass': [],
        'Orbit': [],
        'Customer': [],
        'Launch outcome': [],
        'Booster landing': []
    }
    
    extracted_row = 0
    
    # 提取每个表格
    for table in soup.find_all('table', "wikitable plainrowheaders collapsible"):
        for rows in table.find_all("tr"):
            if rows.th:
                flight_number = rows.th.string.strip() if rows.th.string else ''
                flag = flight_number.isdigit()
            else:
                flag = False
            
            row = rows.find_all('td')
            if flag:
                extracted_row += 1
                
                # 填充 launch_dict
                launch_dict['Flight No.'].append(flight_number)
                
                datatimelist = date_time(row[0].text)
                date = datatimelist[0].strip(',')
                launch_dict['Date'].append(date)
                
                time = datatimelist[1]
                launch_dict['Time'].append(time)
                
                bv = booster_version(row[1].text)
                launch_dict['Version, Booster'].append(bv)
                
                launch_site = row[2].a.string if row[2].a else ''
                launch_dict['Launch site'].append(launch_site)
                
                payload = row[3].a.string if row[3].a else ''
                launch_dict['Payload'].append(payload)
                
                payload_mass = get_mass(row[4].text)
                launch_dict['Payload mass'].append(payload_mass)
                
                orbit = row[5].a.string if row[5].a else ''
                launch_dict['Orbit'].append(orbit)
                
                customer = row[6].a.string if row[6].a else ''
                launch_dict['Customer'].append(customer)
                
                launch_outcome = list(row[7].strings)[0] if row[7].strings else ''
                launch_dict['Launch outcome'].append(launch_outcome)
                
                booster_landing = landing_status(row[8].text)
                launch_dict['Booster landing'].append(booster_landing)

    # 创建 Pandas DataFrame
    launch_df = pd.DataFrame(launch_dict)

    # 打印数据框架的前几行
    print("\n创建的数据框架:")
    print(launch_df.head())  # 打印前5行

else:
    print(f"请求失败，状态码: {response.status_code}")


创建的数据框架:
  Flight No. Date      Time      Version, Booster Launch site  \
0          1    4      June  F9 v1.0[7]B0003.1[8]       CCAFS   
1          2    8  December  F9 v1.0[7]B0004.1[8]       CCAFS   
2          3   22       May  F9 v1.0[7]B0005.1[8]       CCAFS   
3          4    8   October  F9 v1.0[7]B0006.1[8]       CCAFS   
4          5    1     March  F9 v1.0[7]B0007.1[8]       CCAFS   

                                Payload           Payload mass Orbit Customer  \
0  Dragon Spacecraft Qualification Unit                          LEO   SpaceX   
1                                Dragon                          LEO     NASA   
2                                Dragon  525 kg (1,157 lb)[19]   LEO     NASA   
3                          SpaceX CRS-1   4,700 kg (10,400 lb)   LEO     NASA   
4                          SpaceX CRS-2   4,877 kg (10,752 lb)   LEO     NASA   

  Launch outcome            Booster landing  
0      Success\n  Failure[9][10](parachute)  
1        Success  Fa

<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


In [4]:
# 过滤掉猎鹰 1 号的发射记录
launch_df_filtered = launch_df[launch_df['Flight No.'] != 'Falcon 1']

# 计算猎鹰 9 号的发射次数
falcon_9_launches_count = launch_df_filtered[launch_df_filtered['Version, Booster'].str.contains('Falcon 9')].shape[0]

print(f"除去猎鹰 1 号的发射后，猎鹰 9 号的发射次数为: {falcon_9_launches_count}")

除去猎鹰 1 号的发射后，猎鹰 9 号的发射次数为: 0


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


In [5]:
# 过滤掉猎鹰 1 号的发射记录
launch_df_filtered = launch_df[launch_df['Flight No.'] != 'Falcon 1']

# 计算不包含猎鹰 9 号的发射次数
not_falcon_9_launches_count = launch_df_filtered[~launch_df_filtered['Version, Booster'].str.contains('Falcon 9')].shape[0]

print(f"除去猎鹰 1 号的发射后，不包含猎鹰 9 号的发射次数为: {not_falcon_9_launches_count}")

除去猎鹰 1 号的发射后，不包含猎鹰 9 号的发射次数为: 121


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
